# Waymo Open Dataset v2 — YOLO Training

End-to-end notebook for exporting Waymo data to YOLO format and training
a 2-D object detector with [Ultralytics YOLOv8](https://docs.ultralytics.com).

**Sections**
1. [Setup](#1.-Setup)
2. [Export YOLO Dataset](#2.-Export-YOLO-Dataset)
3. [Inspect Exported Data](#3.-Inspect-Exported-Data)
4. [Train YOLOv8](#4.-Train-YOLOv8)
5. [Evaluate on Validation Split](#5.-Evaluate-on-Validation-Split)
6. [Run Inference](#6.-Run-Inference)

**Prerequisites**
```bash
# Authenticate with GCS
gcloud auth application-default login

# Install Ultralytics (not in requirements.txt — optional training dep)
pip install ultralytics
```

---
## 1. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('.')), ''))

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import yaml
from pathlib import Path
from IPython.display import display, Image as IPImage

from modules.waymo_open_dataset import ToolKit, CAMERA_NAMES, YOLO_CLASS_NAMES
from modules import visualize as viz

%matplotlib inline
plt.rcParams['figure.dpi'] = 110
print('Imports OK')

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────
YOLO_DIR   = './yolo_dataset'   # root of the exported YOLO dataset
TRAIN_SEGS = 5                  # number of training segments to export
VAL_SEGS   = 2                  # number of validation segments to export

# YOLOv8 model variant: n(ano) | s(mall) | m(edium) | l(arge) | x(large)
MODEL_SIZE = 'n'                # start with nano for speed; upgrade for accuracy
EPOCHS     = 50
IMG_SIZE   = 640                # standard YOLO input resolution
BATCH      = 16
# ──────────────────────────────────────────────────────────────────────────

print(f'YOLO dataset root : {os.path.abspath(YOLO_DIR)}')
print(f'Model             : yolov8{MODEL_SIZE}.pt')
print(f'Classes           : {YOLO_CLASS_NAMES}')

---
## 2. Export YOLO Dataset

Streams data directly from GCS — no full download required.  
Each `export_yolo()` call processes one segment (~20 frames × 5 cameras = ~100 images).

In [ ]:
# ── Export training split ──────────────────────────────────────────────────
toolkit = ToolKit(split='training')
train_segments = toolkit.list_segments()
print(f'Available training segments: {len(train_segments)}')

for i, seg in enumerate(train_segments[:TRAIN_SEGS]):
    print(f'  [{i+1}/{TRAIN_SEGS}] {seg[:50]}…', end=' ', flush=True)
    toolkit.assign_segment(seg)
    toolkit.export_yolo(output_dir=YOLO_DIR, yolo_split='train')
    print('done')

print('\nTraining export complete.')

In [ ]:
# ── Export validation split ────────────────────────────────────────────────
toolkit_val = ToolKit(split='validation')
val_segments = toolkit_val.list_segments()
print(f'Available validation segments: {len(val_segments)}')

for i, seg in enumerate(val_segments[:VAL_SEGS]):
    print(f'  [{i+1}/{VAL_SEGS}] {seg[:50]}…', end=' ', flush=True)
    toolkit_val.assign_segment(seg)
    toolkit_val.export_yolo(output_dir=YOLO_DIR, yolo_split='val')
    print('done')

print('\nValidation export complete.')

---
## 3. Inspect Exported Data

In [ ]:
# ── Dataset statistics ─────────────────────────────────────────────────────
yolo_root = Path(YOLO_DIR)

for split in ('train', 'val'):
    img_dir = yolo_root / 'images' / split
    lbl_dir = yolo_root / 'labels' / split
    if not img_dir.exists():
        continue
    n_imgs  = len(list(img_dir.glob('*.jpg')))
    n_lbls  = len(list(lbl_dir.glob('*.txt')))
    n_boxes = sum(
        len(p.read_text().strip().splitlines())
        for p in lbl_dir.glob('*.txt')
        if p.read_text().strip()
    )
    print(f'{split:5s}  images={n_imgs:5d}  labels={n_lbls:5d}  boxes={n_boxes:7d}')

# Print dataset.yaml
yaml_path = yolo_root / 'dataset.yaml'
if yaml_path.exists():
    print()
    print('dataset.yaml:')
    print(yaml_path.read_text())

In [ ]:
# ── Class distribution ─────────────────────────────────────────────────────
from collections import Counter

counts = Counter()
lbl_dir = yolo_root / 'labels' / 'train'
for lbl_file in lbl_dir.glob('*.txt'):
    for line in lbl_file.read_text().strip().splitlines():
        if line:
            counts[int(line.split()[0])] += 1

labels = [YOLO_CLASS_NAMES[i] for i in sorted(counts)]
values = [counts[i] for i in sorted(counts)]
colors = [list(viz.LABEL_COLORS_RGB.values())[i+1] for i in sorted(counts)]

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(labels, values, color=colors, edgecolor='white', linewidth=0.6)
ax.bar_label(bars, fmt='%d', padding=3, fontsize=9)
ax.set_title('YOLO export — class distribution (train split)')
ax.set_ylabel('Box count')
ax.set_xlabel('Class')
plt.tight_layout()
plt.show()

In [ ]:
# ── Visual sanity check: draw YOLO labels on a random training image ───────
import random

img_paths = sorted((yolo_root / 'images' / 'train').glob('*.jpg'))
sample_img_path = random.choice(img_paths)
sample_lbl_path = (yolo_root / 'labels' / 'train' /
                   sample_img_path.with_suffix('.txt').name)

img = cv2.imread(str(sample_img_path))
h, w = img.shape[:2]

for line in sample_lbl_path.read_text().strip().splitlines():
    if not line:
        continue
    cls, cx_n, cy_n, bw_n, bh_n = map(float, line.split())
    cls = int(cls)
    cx, cy = cx_n * w, cy_n * h
    bw, bh = bw_n * w, bh_n * h
    x1, y1 = int(cx - bw / 2), int(cy - bh / 2)
    x2, y2 = int(cx + bw / 2), int(cy + bh / 2)
    label_name = YOLO_CLASS_NAMES[cls]
    # Map class name to visualize color key
    color_key = f'TYPE_{label_name.upper()}'
    color = viz.LABEL_COLORS_BGR.get(color_key, (200, 200, 200))
    cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
    cv2.putText(img, label_name, (x1, max(y1 - 5, 10)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1, cv2.LINE_AA)

fig, ax = plt.subplots(figsize=(12, 6))
ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
ax.set_title(f'Sanity check: {sample_img_path.name}')
ax.axis('off')
plt.tight_layout()
plt.show()

---
## 4. Train YOLOv8

Uses [Ultralytics](https://docs.ultralytics.com) — the standard plug-and-play YOLO wrapper.  
`yolov8n` trains fast on CPU/GPU and is a good starting point; swap for `yolov8m` or `yolov8l` on Colab A100.

> **Colab tip:** Mount Drive and set `YOLO_DIR` + `PROJECT_DIR` to `/content/drive/MyDrive/...`
> so checkpoints survive session restarts.

In [ ]:
try:
    from ultralytics import YOLO
except ImportError:
    raise ImportError(
        'ultralytics not installed. Run: pip install ultralytics'
    )

In [ ]:
PROJECT_DIR = './runs/waymo'    # where training artefacts are saved
RUN_NAME    = f'yolov8{MODEL_SIZE}_waymo'

model = YOLO(f'yolov8{MODEL_SIZE}.pt')  # auto-downloads pretrained weights

results = model.train(
    data    = str(yaml_path),
    epochs  = EPOCHS,
    imgsz   = IMG_SIZE,
    batch   = BATCH,
    project = PROJECT_DIR,
    name    = RUN_NAME,
    exist_ok= True,             # resume if interrupted
    verbose = True,
)

In [ ]:
# ── Training curves ────────────────────────────────────────────────────────
run_dir = Path(PROJECT_DIR) / RUN_NAME
results_csv = run_dir / 'results.csv'

if results_csv.exists():
    import pandas as pd
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    # Box loss
    axes[0].plot(df['epoch'], df['train/box_loss'], label='train')
    axes[0].plot(df['epoch'], df['val/box_loss'],   label='val')
    axes[0].set_title('Box loss')
    axes[0].set_xlabel('Epoch')
    axes[0].legend()

    # Class loss
    axes[1].plot(df['epoch'], df['train/cls_loss'], label='train')
    axes[1].plot(df['epoch'], df['val/cls_loss'],   label='val')
    axes[1].set_title('Class loss')
    axes[1].set_xlabel('Epoch')
    axes[1].legend()

    # mAP50
    axes[2].plot(df['epoch'], df['metrics/mAP50(B)'], color='green')
    axes[2].set_title('mAP@50')
    axes[2].set_xlabel('Epoch')

    plt.suptitle(f'Training run: {RUN_NAME}', fontsize=11)
    plt.tight_layout()
    plt.show()
else:
    print('results.csv not found — run training first.')

---
## 5. Evaluate on Validation Split

In [ ]:
best_weights = run_dir / 'weights' / 'best.pt'

if best_weights.exists():
    model_eval = YOLO(str(best_weights))
    metrics = model_eval.val(data=str(yaml_path), imgsz=IMG_SIZE, verbose=True)

    print(f'\nmAP50         : {metrics.box.map50:.4f}')
    print(f'mAP50-95      : {metrics.box.map:.4f}')
    print(f'Per-class AP50: {dict(zip(YOLO_CLASS_NAMES, metrics.box.ap50))}')
else:
    print('No best.pt found — complete training first.')

---
## 6. Run Inference

Predict on a random validation image and visualise detections vs ground-truth side-by-side.

In [ ]:
val_imgs = sorted((yolo_root / 'images' / 'val').glob('*.jpg'))
if not val_imgs:
    raise RuntimeError('No validation images found. Export val split first.')

sample = random.choice(val_imgs)
sample_lbl = (yolo_root / 'labels' / 'val' /
              sample.with_suffix('.txt').name)

model_infer = YOLO(str(best_weights))
preds = model_infer.predict(str(sample), imgsz=IMG_SIZE, conf=0.25, verbose=False)
pred = preds[0]

orig = cv2.imread(str(sample))
h, w = orig.shape[:2]

# -- Ground truth --
gt_img = orig.copy()
for line in sample_lbl.read_text().strip().splitlines():
    if not line:
        continue
    cls, cx_n, cy_n, bw_n, bh_n = map(float, line.split())
    cls = int(cls)
    cx, cy = cx_n * w, cy_n * h
    bw, bh = bw_n * w, bh_n * h
    x1, y1 = int(cx - bw / 2), int(cy - bh / 2)
    x2, y2 = int(cx + bw / 2), int(cy + bh / 2)
    label_name = YOLO_CLASS_NAMES[cls]
    color = viz.LABEL_COLORS_BGR.get(f'TYPE_{label_name.upper()}', (200, 200, 200))
    cv2.rectangle(gt_img, (x1, y1), (x2, y2), color, 2)
    cv2.putText(gt_img, label_name, (x1, max(y1 - 5, 10)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1, cv2.LINE_AA)

# -- Predictions --
pred_img = orig.copy()
for box in pred.boxes:
    cls = int(box.cls)
    conf = float(box.conf)
    x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
    label_name = YOLO_CLASS_NAMES[cls]
    color = viz.LABEL_COLORS_BGR.get(f'TYPE_{label_name.upper()}', (200, 200, 200))
    cv2.rectangle(pred_img, (x1, y1), (x2, y2), color, 2)
    cv2.putText(pred_img, f'{label_name} {conf:.2f}',
                (x1, max(y1 - 5, 10)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1, cv2.LINE_AA)

# -- Side-by-side --
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].imshow(cv2.cvtColor(gt_img,   cv2.COLOR_BGR2RGB))
axes[0].set_title('Ground truth')
axes[0].axis('off')
axes[1].imshow(cv2.cvtColor(pred_img, cv2.COLOR_BGR2RGB))
axes[1].set_title(f'YOLOv8{MODEL_SIZE} predictions (conf≥0.25)')
axes[1].axis('off')
plt.suptitle(sample.name, fontsize=9, color='gray')
plt.tight_layout()
plt.show()